<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat%2Fp2-task-router/notebooks/p2_router_colab_seed2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧭 Thực Nghiệm P2: Bộ Định Tuyến Tác Vụ (Task-Aware Router — P2a & P2b)
### Protocol v1 Track A (FewRel 8 tasks × 10 relations, strict 5-shot, Rehearsal-Free $M=0$, Seed 2021)

Notebook này thực thi hoàn chỉnh hai cơ chế định tuyến tác vụ của đề tài **TAPTA** trên môi trường **Google Colab GPU (CUDA)** mà **không sử dụng Oracle Task ID lúc kiểm thử**:
- **P2a (`E009` — Centroid Cosine Router):** Định tuyến không cần huấn luyện dựa trên tổ hợp tâm tác vụ ($\mu_t$) và tâm con quan hệ ($\nu_{t,c}$) trong không gian BERT đóng băng.
- **P2b (`E010` — Lightweight Learned MLP Cosine Router):** Mạng MLP 2 tầng + Cosine Task Keys huấn luyện kèm tái hiện thống kê đặc trưng Gaussian đóng băng ($M=0$, không lưu câu thô).
- **Mốc tham chiếu:** Cận dưới **P0** (`52.83%`) và Cận trên lý tưởng **P1** (`70.14%`).

## 1. Kiểm tra Môi trường GPU (NVIDIA CUDA Preflight Check)

In [1]:
import torch
print('=' * 60)
print('CUDA / GPU PREFLIGHT CHECK')
print('=' * 60)
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device:     {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    raise RuntimeError('Vui lòng bật GPU trên Google Colab: Runtime -> Change runtime type -> T4 GPU!')

CUDA / GPU PREFLIGHT CHECK
CUDA Available: True
GPU Device:     Tesla T4
Total VRAM:     14.56 GB


## 2. Clone / Cập nhật Repository `research-code-lab`

In [2]:
import os
REPO_URL = 'https://github.com/vinhlq2512/research-code-lab.git'
BRANCH = 'feat/p2-task-router'
REPO_DIR = '/content/research-code-lab'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd /content/research-code-lab
!pip install -q transformers pyyaml

Cloning into '/content/research-code-lab'...
remote: Enumerating objects: 3007, done.
remote: Counting objects: 100% (3007/3007), done.
remote: Compressing objects: 100% (2144/2144), done.
remote: Total 3007 (delta 823), reused 2853 (delta 684), pack-reused 0 (from 0)
Receiving objects: 100% (3007/3007), 17.09 MiB | 11.80 MiB/s, done.
Resolving deltas: 100% (823/823), done.
/content/research-code-lab


## 3. Chạy Thực Nghiệm P2a: Centroid Cosine Router (Training-Free)

In [3]:
!python3 experiments/run_p2.py \
    --config configs/p2a_centroid_router_fewrel_5shot_seed2021.yaml \
    --device cuda

Continual Relation Extraction: P2a (p2a_centroid_router)
Description: Proposed Method P2a: Frozen BERT with task-dedicated prompt pool routed at test time via training-free Cosine Centroid & Relation Sub-Centroid similarity (Zero Oracle Task ID)
Router Type: centroid
Seed:        2021
Results Dir: /content/research-code-lab/runs/p2a_centroid_router/fewrel_5shot/seed_2021

[1/4] Loaded Protocol v1 Dataset: 8 continual tasks.
[02:18:49] [INFO] HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
[02:18:49] [INFO] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"
[02:18:49] [INFO] HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"
[02:18:49] [INFO] HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/86b5e0934494bd15c9632b12f734a8a67f723594/config.json "HTTP/1.1 200 OK"
[02:18:49] [WARNING] Warning: You are sending unauthenticated requests to the 

## 4. Chạy Thực Nghiệm P2b: Lightweight Learned MLP Cosine Router ($M=0$)

In [4]:
!python3 experiments/run_p2.py \
    --config configs/p2b_learned_router_fewrel_5shot_seed2021.yaml \
    --device cuda

Continual Relation Extraction: P2b (p2b_learned_router)
Description: Proposed Method P2b: Frozen BERT with task-dedicated prompt pool routed at test time via a 2-layer MLP Cosine Router trained with stationary frozen-space Gaussian feature statistics (Zero Oracle Task ID, M=0)
Router Type: learned_mlp
Seed:        2021
Results Dir: /content/research-code-lab/runs/p2b_learned_router/fewrel_5shot/seed_2021

[1/4] Loaded Protocol v1 Dataset: 8 continual tasks.
[02:23:26] [INFO] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"
[02:23:26] [INFO] HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"
[02:23:26] [INFO] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
[02:23:26] [INFO] HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/a

## 5. Tổng Hợp Đối Sánh Khung Kẹp (P0 vs P2a vs P2b vs P1) & Tải Kết Quả

In [5]:
import json
from pathlib import Path

p2a_path = Path('runs/p2a_centroid_router/fewrel_5shot/seed_2021/metrics.json')
p2b_path = Path('runs/p2b_learned_router/fewrel_5shot/seed_2021/metrics.json')

print('=' * 85)
print(f"{'Model / Experiment':<30} | {'Router Acc':>10} | {'Proto Acc':>10} | {'Gated Acc':>10} | {'Gap Closed':>10}")
print('-' * 85)
print(f"{'P0 (Shared Prompt - Floor)':<30} | {'N/A':>10} | {'52.83%':>10} | {'N/A':>10} | {'0.00%':>10}")

for label, path in [('P2a (Centroid Cosine Router)', p2a_path), ('P2b (Learned MLP Router)', p2b_path)]:
    if path.exists():
        m = json.loads(path.read_text())
        r_acc = f"{m['router_accuracy_whole']*100:.2f}%"
        p_acc = f"{m['prototype_accuracy_whole']*100:.2f}%"
        g_acc = f"{m['routed_prototype_accuracy_whole']*100:.2f}%"
        gap = f"{m['oracle_gap_closed_pct']:.2f}%"
        print(f"{label:<30} | {r_acc:>10} | {p_acc:>10} | {g_acc:>10} | {gap:>10}")

print(f"{'P1 (Oracle Prompt - Ceiling)':<30} | {'100.00%':>10} | {'70.14%':>10} | {'82.75%':>10} | {'100.00%':>10}")
print('=' * 85)

!zip -r /content/p2_router_results_seed2021.zip runs/p2a_centroid_router runs/p2b_learned_router
print('\nĐã đóng gói kết quả tại: /content/p2_router_results_seed2021.zip')

Model / Experiment             | Router Acc |  Proto Acc |  Gated Acc | Gap Closed
-------------------------------------------------------------------------------------
P0 (Shared Prompt - Floor)     |        N/A |     52.83% |        N/A |      0.00%
P2a (Centroid Cosine Router)   |     64.69% |     59.27% |     59.35% |     37.69%
P2b (Learned MLP Router)       |     50.85% |     56.04% |     46.25% |     18.56%
P1 (Oracle Prompt - Ceiling)   |    100.00% |     70.14% |     82.75% |    100.00%
  adding: runs/p2a_centroid_router/ (stored 0%)
  adding: runs/p2a_centroid_router/fewrel_5shot/ (stored 0%)
  adding: runs/p2a_centroid_router/fewrel_5shot/seed_2021/ (stored 0%)
  adding: runs/p2a_centroid_router/fewrel_5shot/seed_2021/router_confusion_matrix.csv (deflated 60%)
  adding: runs/p2a_centroid_router/fewrel_5shot/seed_2021/performance_matrix.csv (deflated 54%)
  adding: runs/p2a_centroid_router/fewrel_5shot/seed_2021/run.log (deflated 85%)
  adding: runs/p2a_centroid_router/fewrel